### Setup

In [1]:
import importlib
import polars as pl

pl.Config.set_tbl_rows(25)

from pathlib import Path

REPO_DIR = Path('.').resolve().parent
DEV_DATA_DIR = REPO_DIR / 'trio-dev-data'

import sys
sys.path.insert(0, str(REPO_DIR / 'src'))
sys.path.insert(0, str(REPO_DIR / 'src' / 'util'))

In [2]:
KID_ID = 'NA12878'
DAD_ID = 'NA12891'
MOM_ID = 'NA12892'

VCF_TRIO_PHASED = str(DEV_DATA_DIR / 'output' / 'trio-phasing' / 'CEPH-1463.joint.GRCh38.deepvariant.glnexus.phased.vcf.gz')

def blocks_tsv_path(uid):
    return str(DEV_DATA_DIR / 'output' / 'trio-phasing' / f'CEPH-1463.joint.GRCh38.deepvariant.glnexus.phased.{uid}.blocks.tsv')

BLOCKS_TSV_KID = blocks_tsv_path(KID_ID)
BLOCKS_TSV_DAD = blocks_tsv_path(DAD_ID)
BLOCKS_TSV_MOM = blocks_tsv_path(MOM_ID)

In [3]:
METH_BASE_DIR = DEV_DATA_DIR / 'output' / 'dna-methylation'

def meth_bed_paths(uid):
    """Return (count_hap1, count_hap2, model_hap1, model_hap2) paths for a sample."""
    count_dir = METH_BASE_DIR / 'CEPH1463.GRCh38.hifi.count.pedmec-phased'
    model_dir = METH_BASE_DIR / 'CEPH1463.GRCh38.hifi.model.pedmec-phased'
    return (
        str(count_dir / f'{uid}.GRCh38.haplotagged.hap1.bed.gz'),
        str(count_dir / f'{uid}.GRCh38.haplotagged.hap2.bed.gz'),
        str(model_dir / f'{uid}.GRCh38.haplotagged.hap1.bed.gz'),
        str(model_dir / f'{uid}.GRCh38.haplotagged.hap2.bed.gz'),
    )

KID_METH = meth_bed_paths(KID_ID)
DAD_METH = meth_bed_paths(DAD_ID)
MOM_METH = meth_bed_paths(MOM_ID)

### Get phase blocks for each individual (pre-computed by `run-whatshap.sh`)

In [4]:
import phasing_trio
importlib.reload(phasing_trio)
from phasing_trio import get_phase_blocks

DF_BLOCKS_KID = get_phase_blocks(BLOCKS_TSV_KID, KID_ID)
DF_BLOCKS_DAD = get_phase_blocks(BLOCKS_TSV_DAD, DAD_ID)
DF_BLOCKS_MOM = get_phase_blocks(BLOCKS_TSV_MOM, MOM_ID)

print(f'Kid phase blocks: {len(DF_BLOCKS_KID)}')
print(f'Dad phase blocks: {len(DF_BLOCKS_DAD)}')
print(f'Mom phase blocks: {len(DF_BLOCKS_MOM)}')

Kid phase blocks: 1
Dad phase blocks: 1
Mom phase blocks: 1


In [5]:
DF_BLOCKS_KID

chrom,start,end,phase_block_id,num_variants
str,i64,i64,str,i64
"""chr1""",1000334,1999756,"""1000335""",802


In [6]:
DF_BLOCKS_DAD

chrom,start,end,phase_block_id,num_variants
str,i64,i64,str,i64
"""chr1""",1000334,1999860,"""1000335""",1231


In [7]:
DF_BLOCKS_MOM

chrom,start,end,phase_block_id,num_variants
str,i64,i64,str,i64
"""chr1""",1000334,1993898,"""1000335""",949


### Get paired parent-child alleles from the WhatsHap trio-phased VCF (PedMEC algorithm)

In [8]:
import phasing_trio
importlib.reload(phasing_trio)
from phasing_trio import get_all_phasing

DF_KID_DAD, DF_KID_MOM = get_all_phasing(
    VCF_TRIO_PHASED, KID_ID, DAD_ID, MOM_ID,
    DF_BLOCKS_KID, DF_BLOCKS_DAD, DF_BLOCKS_MOM,
)

print(f'Dad-het sites: {len(DF_KID_DAD)}')
print(f'Mom-het sites: {len(DF_KID_MOM)}')

Dad-het sites: 1058
Mom-het sites: 801


In [9]:
DF_KID_DAD

chrom,start,end,REF,ALT,kid_allele_pat,dad_allele_A,dad_allele_B,start_phase_block_kid,end_phase_block_kid,start_phase_block_dad,end_phase_block_dad
str,i64,i64,str,str,str,str,str,i64,i64,i64,i64
"""chr1""",1000334,1000335,"""C""","""T""","""1""","""1""","""0""",1000334,1999756,1000334,1999860
"""chr1""",1000730,1000731,"""C""","""T""","""0""","""0""","""1""",1000334,1999756,1000334,1999860
"""chr1""",1002744,1002745,"""G""","""A""","""1""","""1""","""0""",1000334,1999756,1000334,1999860
"""chr1""",1004624,1004625,"""A""","""G""","""1""","""1""","""0""",1000334,1999756,1000334,1999860
"""chr1""",1004715,1004716,"""C""","""T""","""0""","""0""","""1""",1000334,1999756,1000334,1999860
"""chr1""",1004822,1004823,"""G""","""A""","""0""","""0""","""1""",1000334,1999756,1000334,1999860
"""chr1""",1005903,1005904,"""C""","""T""","""1""","""1""","""0""",1000334,1999756,1000334,1999860
"""chr1""",1005953,1005954,"""G""","""A""","""1""","""1""","""0""",1000334,1999756,1000334,1999860
"""chr1""",1006158,1006159,"""C""","""T""","""1""","""1""","""0""",1000334,1999756,1000334,1999860


In [10]:
DF_KID_MOM

chrom,start,end,REF,ALT,kid_allele_mat,mom_allele_C,mom_allele_D,start_phase_block_kid,end_phase_block_kid,start_phase_block_mom,end_phase_block_mom
str,i64,i64,str,str,str,str,str,i64,i64,i64,i64
"""chr1""",1000334,1000335,"""C""","""T""","""0""","""0""","""1""",1000334,1999756,1000334,1993898
"""chr1""",1000730,1000731,"""C""","""T""","""1""","""1""","""0""",1000334,1999756,1000334,1993898
"""chr1""",1002744,1002745,"""G""","""A""","""0""","""0""","""1""",1000334,1999756,1000334,1993898
"""chr1""",1004624,1004625,"""A""","""G""","""0""","""0""","""1""",1000334,1999756,1000334,1993898
"""chr1""",1004715,1004716,"""C""","""T""","""1""","""1""","""0""",1000334,1999756,1000334,1993898
"""chr1""",1005903,1005904,"""C""","""T""","""0""","""0""","""1""",1000334,1999756,1000334,1993898
"""chr1""",1005953,1005954,"""G""","""A""","""0""","""0""","""1""",1000334,1999756,1000334,1993898
"""chr1""",1006158,1006159,"""C""","""T""","""0""","""0""","""1""",1000334,1999756,1000334,1993898
"""chr1""",1008306,1008307,"""G""","""C""","""0""","""0""","""1""",1000334,1999756,1000334,1993898


### Construct a Hap Map, consisting of intervals ("hap-map blocks") in which haplotypes in the kid are mapped to haplotypes in the parents

Labeling convention:
- **A** = dad's hap1, **B** = dad's hap2 (fixed)
- **C** = mom's hap1, **D** = mom's hap2 (fixed)
- Kid's hap1 (paternal) matches A or B per block; kid's hap2 (maternal) matches C or D per block

Note that A, B, C, D are only defined within hap-map blocks, not chromosome-wide, as is the case for the pedigree-wise workflow

In [11]:
import hap_map_trio
importlib.reload(hap_map_trio)
from hap_map_trio import get_hap_map

DF_HAP_MAP_PAT, DF_HAP_MAP_MAT, DF_MISMATCH_PAT, DF_MISMATCH_MAT = get_hap_map(DF_KID_DAD, DF_KID_MOM)

In [12]:
DF_HAP_MAP_PAT

chrom,start,end,paternal_haplotype,haplotype_concordance,num_het_SNVs_in_parent
str,i64,i64,str,f64,i64
"""chr1""",1000334,1999756,"""A""",0.999055,1058


In [13]:
DF_MISMATCH_PAT

chrom,start,end,REF,ALT
str,i64,i64,str,str
"""chr1""",1076638,1076639,"""G""","""A"""


In [14]:
DF_HAP_MAP_MAT

chrom,start,end,maternal_haplotype,haplotype_concordance,num_het_SNVs_in_parent
str,i64,i64,str,f64,i64
"""chr1""",1000334,1993898,"""C""",0.982522,801


In [15]:
DF_MISMATCH_MAT

chrom,start,end,REF,ALT
str,i64,i64,str,str
"""chr1""",1289587,1289588,"""T""","""C"""
"""chr1""",1289720,1289721,"""T""","""C"""
"""chr1""",1289740,1289741,"""T""","""C"""
"""chr1""",1289903,1289904,"""T""","""C"""
"""chr1""",1302448,1302449,"""G""","""A"""
"""chr1""",1308548,1308549,"""G""","""T"""
"""chr1""",1317019,1317020,"""C""","""T"""
"""chr1""",1324156,1324157,"""G""","""A"""
"""chr1""",1324165,1324166,"""G""","""A"""


### Get HiFi DNA methylation levels (both count-based and model-based) in kid, dad, and mom at CpG sites phased to hap1/hap2

In [16]:
import get_meth_hap1_hap2
importlib.reload(get_meth_hap1_hap2)
from get_meth_hap1_hap2 import read_meth_hap1_hap2

DF_METH_COUNT_HAP1_HAP2_KID = read_meth_hap1_hap2(
    pb_cpg_tool_mode='count',
    bed_hap1=KID_METH[0],
    bed_hap2=KID_METH[1],
)
DF_METH_COUNT_HAP1_HAP2_KID

chrom,start,end,total_read_count_hap1,methylation_level_hap1,total_read_count_hap2,methylation_level_hap2
str,i64,i64,i64,f64,i64,f64
"""chr1""",990862,990863,10,0.7,null,null
"""chr1""",990869,990870,10,0.7,null,null
"""chr1""",990914,990915,10,0.7,null,null
"""chr1""",990924,990925,10,0.7,null,null
"""chr1""",990948,990949,11,0.818,null,null
"""chr1""",990956,990957,11,0.818,null,null
"""chr1""",991280,991281,11,0.818,null,null
"""chr1""",991431,991432,12,0.75,null,null
"""chr1""",991460,991461,12,0.75,null,null


In [17]:
DF_METH_MODEL_HAP1_HAP2_KID = read_meth_hap1_hap2(
    pb_cpg_tool_mode='model',
    bed_hap1=KID_METH[2],
    bed_hap2=KID_METH[3],
)

In [18]:
DF_METH_COUNT_HAP1_HAP2_DAD = read_meth_hap1_hap2(
    pb_cpg_tool_mode='count',
    bed_hap1=DAD_METH[0],
    bed_hap2=DAD_METH[1],
)

In [19]:
DF_METH_MODEL_HAP1_HAP2_DAD = read_meth_hap1_hap2(
    pb_cpg_tool_mode='model',
    bed_hap1=DAD_METH[2],
    bed_hap2=DAD_METH[3],
)

In [20]:
DF_METH_COUNT_HAP1_HAP2_MOM = read_meth_hap1_hap2(
    pb_cpg_tool_mode='count',
    bed_hap1=MOM_METH[0],
    bed_hap2=MOM_METH[1],
)

In [21]:
DF_METH_MODEL_HAP1_HAP2_MOM = read_meth_hap1_hap2(
    pb_cpg_tool_mode='model',
    bed_hap1=MOM_METH[2],
    bed_hap2=MOM_METH[3],
)

### Phase DNA methylation levels in trio to parent haplotypes

In [22]:
import phase_meth_to_parent_haps
importlib.reload(phase_meth_to_parent_haps)
from phase_meth_to_parent_haps import phase_meth_to_parent_haplotypes

DF_METH_PARENT_PHASED = phase_meth_to_parent_haplotypes(
    DF_METH_COUNT_HAP1_HAP2_KID,
    DF_METH_MODEL_HAP1_HAP2_KID,
    DF_METH_COUNT_HAP1_HAP2_DAD,
    DF_METH_MODEL_HAP1_HAP2_DAD,
    DF_METH_COUNT_HAP1_HAP2_MOM,
    DF_METH_MODEL_HAP1_HAP2_MOM,
    DF_HAP_MAP_PAT,
    DF_HAP_MAP_MAT,
)
DF_METH_PARENT_PHASED.filter(pl.col("start_hap_map_block_pat").is_not_null()).head()

chrom,start,end,total_read_count_kid_pat,methylation_level_kid_pat_count,total_read_count_kid_mat,methylation_level_kid_mat_count,methylation_level_kid_pat_model,methylation_level_kid_mat_model,start_hap_map_block_pat,end_hap_map_block_pat,paternal_haplotype,paternal_concordance,num_het_SNVs_in_dad,start_hap_map_block_mat,end_hap_map_block_mat,maternal_haplotype,maternal_concordance,num_het_SNVs_in_mom,total_read_count_dad_A,methylation_level_dad_A_count,total_read_count_dad_B,methylation_level_dad_B_count,methylation_level_dad_A_model,methylation_level_dad_B_model,total_read_count_mom_C,methylation_level_mom_C_count,total_read_count_mom_D,methylation_level_mom_D_count,methylation_level_mom_C_model,methylation_level_mom_D_model
str,i64,i64,i64,f64,i64,f64,f64,f64,i64,i64,str,f64,i64,i64,i64,str,f64,i64,i64,f64,i64,f64,f64,f64,i64,f64,i64,f64,f64,f64
"""chr1""",1000334,1000335,26,0.0,18,0.111,0.025,0.04,1000334,1999756,"""A""",0.999055,1058,1000334,1993898,"""C""",0.982522,801,13,0.077,21,0.0,0.04,0.02,19,0.0,29,0.0,0.032,0.027
"""chr1""",1000343,1000344,26,0.154,18,0.278,0.034,0.074,1000334,1999756,"""A""",0.999055,1058,1000334,1993898,"""C""",0.982522,801,13,0.231,21,0.095,0.031,0.03,19,0.211,29,0.207,0.039,0.03
"""chr1""",1000349,1000350,26,0.231,18,0.278,0.034,0.109,1000334,1999756,"""A""",0.999055,1058,1000334,1993898,"""C""",0.982522,801,13,0.077,21,0.048,0.034,0.028,19,0.105,29,0.103,0.035,0.032
"""chr1""",1000356,1000357,26,0.077,18,0.111,0.037,0.041,1000334,1999756,"""A""",0.999055,1058,1000334,1993898,"""C""",0.982522,801,13,0.077,21,0.0,0.03,0.027,19,0.053,28,0.107,0.033,0.03
"""chr1""",1000358,1000359,26,0.115,18,0.167,0.033,0.036,1000334,1999756,"""A""",0.999055,1058,1000334,1993898,"""C""",0.982522,801,13,0.077,21,0.048,0.036,0.028,19,0.053,28,0.143,0.025,0.034
